In [ ]:
import requests
import pandas as pd
import time

base_url = "https://www.fema.gov/api/open/v2/IndividualsAndHouseholdsProgramValidRegistrations"
years = [2019, 2020, 2021, 2022, 2023]
all_summaries = []

print("Starting Year-by-Year Extraction to prevent RAM crashes...")

for year in years:
    print(f"Processing {year} ")
    # Filter for the specific year
    api_filter = f"$filter=declarationDate gt '{year-1}-12-31T23:59:59Z' and declarationDate lt '{year+1}-01-01T00:00:00Z'"

    # We use $inlinecount=allpages to find out how many records exist for this year
    # But for simplicity, we will fetch in chunks of 10k for each year
    skip = 0
    year_data = []

    while True:
        url = f"{base_url}?{api_filter}&$top=10000&$skip={skip}&$format=json"
        try:
            response = requests.get(url, timeout=30)
            if response.status_code != 200: break

            data = response.json().get('IndividualsAndHouseholdsProgramValidRegistrations', [])
            if not data: break

            # Convert chunk to summary immediately to save RAM
            df_chunk = pd.DataFrame(data)
            df_chunk['haAmount'] = pd.to_numeric(df_chunk['haAmount'], errors='coerce').fillna(0)
            df_chunk['ihpAmount'] = pd.to_numeric(df_chunk['ihpAmount'], errors='coerce').fillna(0)
            df_chunk['Year'] = year

            summary = df_chunk.groupby(['Year', 'damagedStateAbbreviation', 'county']).agg({
                'id': 'count',
                'haAmount': 'sum',
                'ihpAmount': 'sum'
            }).reset_index()

            year_data.append(summary)
            skip += 10000
            print(f"   Fetched {skip} records for {year}...")
            time.sleep(0.5) # Prevent API rate-limiting

        except Exception as e:
            print(f"   Retry needed for {year} at skip {skip}: {e}")
            time.sleep(5)
            continue

    if year_data:
        # Combine all summaries for this specific year
        year_summary = pd.concat(year_data).groupby(['Year', 'damagedStateAbbreviation', 'county']).sum().reset_index()
        all_summaries.append(year_summary)
        print(f"Finished {year}. Summary rows for this year: {len(year_summary)}")

# Final Merge of all years
if all_summaries:
    final_df = pd.concat(all_summaries).reset_index(drop=True)
    final_df.rename(columns={
        'damagedStateAbbreviation': 'State',
        'id': 'total_applicants',
        'haAmount': 'housing_assistance_amount',
        'ihpAmount': 'total_assistance_amount'
    }, inplace=True)

    final_df.to_csv('data/fema_assistance_yearly_summary.csv', index=False)
    print(f"\nFINAL SUCCESS!")
    print(f"Total summarized rows: {len(final_df)}")
    print(f"Years captured: {final_df['Year'].unique()}")

Starting Year-by-Year Extraction to prevent RAM crashes...
Processing 2019 
   Fetched 10000 records for 2019...
   Fetched 20000 records for 2019...
   Fetched 30000 records for 2019...
   Fetched 40000 records for 2019...
   Fetched 50000 records for 2019...
   Fetched 60000 records for 2019...
   Fetched 70000 records for 2019...
   Fetched 80000 records for 2019...
Finished 2019. Summary rows for this year: 161
Processing 2020 
   Fetched 10000 records for 2020...
   Fetched 20000 records for 2020...
   Fetched 30000 records for 2020...
   Fetched 40000 records for 2020...
   Fetched 50000 records for 2020...
   Fetched 60000 records for 2020...
   Fetched 70000 records for 2020...
   Fetched 80000 records for 2020...
   Fetched 90000 records for 2020...
   Fetched 100000 records for 2020...
   Fetched 110000 records for 2020...
   Fetched 120000 records for 2020...
   Fetched 130000 records for 2020...
   Fetched 140000 records for 2020...
   Fetched 150000 records for 2020...
   